# Generate and filter 6-simul pin sets

This notebook does two things:

1. Generate every unordered set of six distinct pin states, excluding `ALL` and `all`.
2. Keep only sets whose 14×12 move matrix has full column rank modulo 12.

The output text files are written to `txt/`.

In [2]:
import json
from itertools import combinations, product
from math import gcd
from pathlib import Path

import numpy as np

OUTPUT_DIR = Path("txt")
OUTPUT_DIR.mkdir(exist_ok=True)


def write_pin_sets(path, pin_sets):
    text = "".join(" ".join(pin_set) + "\n" for pin_set in pin_sets)
    path.write_text(text, encoding="utf-8")

## 1. Generate every six-pin set

There are 14 allowed pin states. Because order does not matter, the number of sets is

$$
\binom{14}{6} = 3003.
$$

In [3]:
pins = {'UL': 0, 'UR': 1, 'DR': 2, 'DL': 3, 'U': 4, '\\': 5, 'L': 6, 'R': 7, '/': 8, 'D': 9, 'dl': 10, 'dr': 11, 'ur': 12, 'ul': 13, 'ALL': 14, 'all': 15}

pin_names = [name for name in pins if name not in {"ALL", "all"}]
all_pin_sets = list(combinations(pin_names, 6))

assert len(pin_names) == 14
assert len(all_pin_sets) == 3003

all_sets_path = OUTPUT_DIR / "6_simul_pin_sets.txt"
write_pin_sets(all_sets_path, all_pin_sets)

print(f"Generated {len(all_pin_sets):,} pin sets.")
print(f"Saved to {all_sets_path}")

Generated 3,003 pin sets.
Saved to txt/6_simul_pin_sets.txt


## 2. Build each 14×12 move matrix

For six selected pin states, take their six columns from `U` and their six columns from `D`:

$$
A = [U_{\mathrm{selected}} \mid D_{\mathrm{selected}}].
$$

This gives 14 rows and 12 columns.

In [4]:
U = np.array([
    [1, 0, 0, 0, 1, 1, 1, 0, 0, 0, 1, 1, 1, 0, 1, 0],
    [1, 1, 0, 0, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 0],
    [0, 1, 0, 0, 1, 0, 0, 1, 1, 0, 1, 1, 0, 1, 1, 0],
    [1, 0, 0, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 0],
    [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0],
    [0, 1, 1, 0, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 0],
    [0, 0, 0, 1, 0, 0, 1, 0, 1, 1, 0, 1, 1, 1, 1, 0],
    [0, 0, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0],
    [0, 0, 1, 0, 0, 1, 0, 1, 0, 1, 1, 0, 1, 1, 1, 0],
    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
], dtype=int)

D = np.array([
    [0, 1, 1, 1, 0, 0, 0, 1, 1, 1, 0, 0, 0, 1, 0, 1],
    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    [1, 0, 1, 1, 0, 1, 1, 0, 0, 1, 0, 0, 1, 0, 0, 1],
    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    [1, 1, 1, 0, 1, 1, 0, 1, 0, 0, 1, 0, 0, 0, 0, 1],
    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    [1, 1, 0, 1, 1, 0, 1, 0, 1, 0, 0, 1, 0, 0, 0, 1],
    [-1, -1, -1, -1, 0, -1, -1, -1, -1, -1, 0, 0, -1, -1, 0, -1],
    [-1, -1, -1, -1, -1, -1, -1, 0, -1, -1, 0, -1, -1, 0, 0, -1],
    [-1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, 0, -1],
    [-1, -1, -1, -1, -1, -1, 0, -1, -1, -1, -1, 0, 0, -1, 0, -1],
    [-1, -1, -1, -1, -1, -1, -1, -1, -1, 0, -1, -1, 0, 0, 0, -1]
], dtype=int)

assert U.shape == D.shape == (14, 16)


def move_matrix(pin_set):
    columns = [pins[name] for name in pin_set]
    return np.concatenate((U[:, columns], D[:, columns]), axis=1)


assert move_matrix(all_pin_sets[0]).shape == (14, 12)

## 3. Remove sets that are not full rank modulo 12

Arithmetic on the clock is modulo 12. Since $12 = 4 \cdot 3$, a 14×12 matrix has full column rank modulo 12 exactly when it has rank 12 modulo both 2 and 3.

Checking ordinary rational rank is insufficient: row operations can create factors of 2 or 3 even though the original entries are only `0`, `1`, and `-1`.

In [5]:
def rank_mod_prime(matrix, prime):
    """Exact Gaussian-elimination rank over GF(prime)."""
    reduced = np.array(matrix, dtype=int) % prime
    row_count, column_count = reduced.shape
    pivot_row = 0

    for column in range(column_count):
        pivot = next(
            (row for row in range(pivot_row, row_count)
             if reduced[row, column] != 0),
            None,
        )
        if pivot is None:
            continue

        reduced[[pivot_row, pivot]] = reduced[[pivot, pivot_row]]
        inverse = pow(int(reduced[pivot_row, column]), -1, prime)
        reduced[pivot_row] = reduced[pivot_row] * inverse % prime

        for row in range(row_count):
            if row != pivot_row:
                factor = reduced[row, column]
                reduced[row] = (reduced[row] - factor * reduced[pivot_row]) % prime

        pivot_row += 1
        if pivot_row == row_count:
            break

    return pivot_row


def is_full_rank_mod_12(pin_set):
    matrix = move_matrix(pin_set)
    return rank_mod_prime(matrix, 2) == 12 and rank_mod_prime(matrix, 3) == 12


full_rank_pin_sets = [
    pin_set for pin_set in all_pin_sets
    if is_full_rank_mod_12(pin_set)
]

full_rank_path = OUTPUT_DIR / "6_simul_pin_sets_full_rank.txt"
write_pin_sets(full_rank_path, full_rank_pin_sets)

print(f"Full-rank pin sets: {len(full_rank_pin_sets):,}")
print(f"Removed: {len(all_pin_sets) - len(full_rank_pin_sets):,}")
print(f"Saved to {full_rank_path}")

Full-rank pin sets: 1,696
Removed: 1,307
Saved to txt/6_simul_pin_sets_full_rank.txt


## 4. Calculate each pin set's conditions

For each of the 1,696 pin sets, calculate the two conditions that must both equal 0 modulo 12. These conditions are used in the next step to create the final groups.

In [6]:
state_names = ("UL", "U", "UR", "L", "C", "R", "DL", "D", "DR", "d", "r", "c", "l", "u")


def left_kernel_basis_mod(matrix, modulus):
    """Return a basis for the left kernel, using elimination modulo 3 or 4."""
    row_count, column_count = matrix.shape
    reduced = np.concatenate(
        (matrix % modulus, np.eye(row_count, dtype=int)),
        axis=1,
    )
    pivot_row = 0

    for column in range(column_count):
        pivot = next(
            (row for row in range(pivot_row, row_count)
             if gcd(int(reduced[row, column]), modulus) == 1),
            None,
        )
        assert pivot is not None

        reduced[[pivot_row, pivot]] = reduced[[pivot, pivot_row]]
        inverse = pow(int(reduced[pivot_row, column]), -1, modulus)
        reduced[pivot_row] = reduced[pivot_row] * inverse % modulus

        for row in range(row_count):
            if row != pivot_row:
                factor = reduced[row, column]
                reduced[row] = (reduced[row] - factor * reduced[pivot_row]) % modulus

        pivot_row += 1

    basis = (-reduced[pivot_row:, column_count:]) % modulus
    assert basis.shape == (2, 14)
    assert np.all(basis @ matrix % modulus == 0)
    return basis


def condition_data(pin_set):
    """Return two display equations and a canonical signature for all conditions."""
    matrix = move_matrix(pin_set)
    basis4 = left_kernel_basis_mod(matrix, 4)
    basis3 = left_kernel_basis_mod(matrix, 3)

    # Chinese remainder theorem: 9 selects mod 4 and 4 selects mod 3.
    basis12 = (9 * basis4 + 4 * basis3) % 12
    signature = tuple(sorted({
        tuple((a * basis12[0] + b * basis12[1]) % 12)
        for a, b in product(range(12), repeat=2)
    }))

    assert len(signature) == 144
    assert np.all(basis12 @ matrix % 12 == 0)
    return basis12, signature


def format_condition(coefficients):
    """Format one coefficient row as an equation equal to zero modulo 12."""
    terms = []
    for coefficient, name in zip(coefficients, state_names):
        coefficient = ((int(coefficient) + 6) % 12) - 6
        if coefficient == 0:
            continue

        magnitude = abs(coefficient)
        term = name if magnitude == 1 else f"{magnitude}{name}"
        if not terms:
            terms.append(f"-{term}" if coefficient < 0 else term)
        else:
            sign = "-" if coefficient < 0 else "+"
            terms.append(f"{sign} {term}")

    return " ".join(terms) + " = 0 (mod 12)"


condition_bases = {}
condition_signatures = {}
known_signatures = {}

for pin_set in full_rank_pin_sets:
    basis, signature = condition_data(pin_set)
    signature = known_signatures.setdefault(signature, signature)
    condition_bases[pin_set] = basis
    condition_signatures[pin_set] = signature

example_pin_set = ("UL", "UR", "DR", "U", "\\", "ur")
print(f"Example: {' '.join(example_pin_set)}")
for row in condition_bases[example_pin_set]:
    print(f"  {format_condition(row)}")
print()
print(f"Conditions computed for {len(condition_signatures):,} pin sets.")

Example: UL UR DR U \ ur
  r - c = 0 (mod 12)
  l - u = 0 (mod 12)

Conditions computed for 1,696 pin sets.


## 5. Group by conditions across the eight orientations

The supplied rotations are used internally to form the 54 classes. The JSON contains each class's conditions, count, and pin sets. The conditions correspond to the first pin set listed in that class.

In [7]:
z = {
    "D": "L", "L": "U", "R": "D", "U": "R",
    "UR": "DR", "UL": "UR", "DR": "DL", "DL": "UL",
    "ul": "ur", "ur": "dr", "dr": "dl", "dl": "ul",
    "/": "\\", "\\": "/",
}
y2 = {
    "D": "U", "L": "L", "R": "R", "U": "D",
    "UR": "ul", "UL": "ur", "DR": "dl", "DL": "dr",
    "ul": "UR", "ur": "UL", "dr": "DL", "dl": "DR",
    "/": "/", "\\": "\\",
}
orientations = [
    ("identity", ()),
    ("z", (z,)),
    ("z2", (z, z)),
    ("z'", (z, z, z)),
    ("y2", (y2,)),
    ("y2 z", (y2, z)),
    ("y2 z2", (y2, z, z)),
    ("y2 z'", (y2, z, z, z)),
]

pin_order = {name: index for index, name in enumerate(pin_names)}


def rotate_pin_set(pin_set, moves):
    for move in moves:
        pin_set = tuple(move[name] for name in pin_set)
    return tuple(sorted(pin_set, key=pin_order.get))


def oriented_pin_sets(pin_set):
    return [
        (name, rotate_pin_set(pin_set, moves))
        for name, moves in orientations
    ]


assert len({
    tuple(rotate_pin_set((name,), moves)[0] for name in pin_names)
    for _, moves in orientations
}) == 8


full_rank_pool = set(full_rank_pin_sets)
assert all(
    rotated in full_rank_pool
    for pin_set in full_rank_pin_sets
    for _, rotated in oriented_pin_sets(pin_set)
)

rotation_groups = {}
for pin_set in full_rank_pin_sets:
    group_key = min(
        condition_signatures[rotated]
        for _, rotated in oriented_pin_sets(pin_set)
    )
    rotation_groups.setdefault(group_key, []).append(pin_set)

classes = []
for number, members in enumerate(rotation_groups.values(), start=1):
    representative = members[0]
    classes.append({
        "class": number,
        "count": len(members),
        "conditions": [
            format_condition(row)
            for row in condition_bases[representative]
        ],
        "pin_sets": [list(pin_set) for pin_set in members],
    })

result = {
    "class_count": len(classes),
    "pin_set_count": sum(item["count"] for item in classes),
    "classes": classes,
}
classes_path = OUTPUT_DIR / "6_simul_pin_sets_classes.json"
with classes_path.open("w", encoding="utf-8") as output:
    output.write("{\n")
    output.write(f"  \"class_count\": {result['class_count']},\n")
    output.write(f"  \"pin_set_count\": {result['pin_set_count']},\n")
    output.write("  \"classes\": [\n")

    for class_index, class_data in enumerate(classes):
        if class_index:
            output.write(",\n")
        output.write("    {\n")
        output.write(f"      \"class\": {class_data['class']},\n")
        output.write(f"      \"count\": {class_data['count']},\n")
        output.write(
            f"      \"conditions\": {json.dumps(class_data['conditions'])},\n"
        )
        output.write("      \"pin_sets\": [\n")
        for pin_set_index, pin_set in enumerate(class_data["pin_sets"]):
            comma = "," if pin_set_index + 1 < len(class_data["pin_sets"]) else ""
            output.write(f"        {json.dumps(pin_set)}{comma}\n")
        output.write("      ]\n")
        output.write("    }")

    output.write("\n  ]\n")
    output.write("}\n")

assert result["class_count"] == 54
assert result["pin_set_count"] == 1696
print(f"Equivalence classes after rotations: {result['class_count']:,}")
print(f"Saved to {classes_path}")

Equivalence classes after rotations: 54
Saved to txt/6_simul_pin_sets_classes.json


## 6. Sort all 720 orders of every pin set

Every permutation is scored separately. Within each class, the orders are sorted by fewest D moves, then fewest pin transitions, then highest number of intuitive moves.

In [8]:
from itertools import permutations
from math import factorial

from chat_intuitive_moves import run_for_order


d_move_pins = {"D", "U", "DR", "DL", "dr", "dl"}
pin_states = {
    "UR": "0100", "UL": "1000", "DR": "0001", "DL": "0010",
    "U": "1100", "D": "0011", "L": "1010", "R": "0101",
    "ur": "1011", "ul": "0111", "dr": "1110", "dl": "1101",
    "/": "0110", "\\": "1001",
}

assert set(pin_states) == set(pin_names)
assert len(set(pin_states.values())) == len(pin_states)

transition_cost = {
    (left, right): sum(a != b for a, b in zip(pin_states[left], pin_states[right]))
    for left in pin_names
    for right in pin_names
}

orders_per_set = factorial(6)
expected_order_count = len(full_rank_pin_sets) * orders_per_set
pin_orders_path = OUTPUT_DIR / "6_simul_pin_orders.json"
total_order_count = 0
best_cases = []

with pin_orders_path.open("w", encoding="utf-8") as output:
    output.write("{\n")
    output.write(f"  \"class_count\": {len(classes)},\n")
    output.write(f"  \"pin_order_count\": {expected_order_count},\n")
    output.write("  \"classes\": [\n")

    for class_index, class_data in enumerate(classes):
        scored_orders = []
        for pin_set in class_data["pin_sets"]:
            d_moves = sum(name in d_move_pins for name in pin_set)
            for order in permutations(pin_set):
                transitions = sum(
                    transition_cost[left, right]
                    for left, right in zip(order, order[1:])
                )
                intuitive_moves = run_for_order(order)[1]
                scored_orders.append((d_moves, transitions, intuitive_moves, order))

        scored_orders.sort(key=lambda item: (item[0], item[1], -item[2]))
        order_count = len(scored_orders)
        total_order_count += order_count
        best_d_moves, best_transitions, best_intuitive, best_order = scored_orders[0]
        best_pin_set = tuple(sorted(best_order, key=pin_order.get))
        best_cases.append({
            "class": class_data["class"],
            "conditions": [
                format_condition(row) for row in condition_bases[best_pin_set]
            ],
            "best_pin_order": list(best_order),
            "d_moves": best_d_moves,
            "transitions": best_transitions,
            "intuitive_moves": best_intuitive,
        })

        if class_index:
            output.write(",\n")
        output.write("    {\n")
        output.write(f"      \"class\": {class_data['class']},\n")
        output.write(f"      \"count\": {order_count},\n")
        output.write(
            f"      \"conditions\": {json.dumps(class_data['conditions'])},\n"
        )
        output.write("      \"pin_orders\": [\n")

        for order_index, (d_moves, transitions, intuitive_moves, order) in enumerate(scored_orders):
            if order_index:
                output.write(",\n")
            record = {
                "order": list(order),
                "d_moves": d_moves,
                "transitions": transitions,
                "intuitive_moves": intuitive_moves,
            }
            output.write("        " + json.dumps(record, separators=(",", ":")))

        output.write("\n      ]\n")
        output.write("    }")

    output.write("\n  ]\n")
    output.write("}\n")

assert total_order_count == expected_order_count
print(f"Sorted pin orders: {total_order_count:,}")
print(f"Saved to {pin_orders_path}")

Sorted pin orders: 1,221,120
Saved to txt/6_simul_pin_orders.json


## 7. Save the best order from each class

The scramble percentage is the exact proportion of uniformly random clock states solvable in at least one of the eight orientations.

In [9]:
from sympy import Matrix
from sympy.matrices.normalforms import smith_normal_form
from sympy.polys.domains import ZZ


def kernel_size_mod_12(rows):
    """Count states satisfying all supplied linear conditions modulo 12."""
    diagonal = smith_normal_form(Matrix(rows), domain=ZZ)
    invariants = [
        abs(int(diagonal[index, index]))
        for index in range(min(diagonal.shape))
        if diagonal[index, index] != 0
    ]
    size = 12 ** (14 - len(invariants))
    for invariant in invariants:
        size *= gcd(invariant, 12)
    return size


def solved_scramble_percentage(order):
    pin_set = tuple(sorted(order, key=pin_order.get))
    unique_systems = {}
    for _, rotated in oriented_pin_sets(pin_set):
        signature = condition_signatures[rotated]
        unique_systems.setdefault(signature, condition_bases[rotated])
    bases = list(unique_systems.values())

    solved = 0
    for subset_size in range(1, len(bases) + 1):
        sign = 1 if subset_size % 2 else -1
        for subset in combinations(bases, subset_size):
            solved += sign * kernel_size_mod_12(np.concatenate(subset, axis=0))

    return 100 * solved / (12 ** 14)


for case in best_cases:
    case["scrambles_solved_percentage"] = round(
        solved_scramble_percentage(case["best_pin_order"]),
        6,
    )

best_cases_path = OUTPUT_DIR / "6_simul_best_pin_orders.json"
with best_cases_path.open("w", encoding="utf-8") as output:
    output.write("{\n")
    output.write(f"  \"class_count\": {len(best_cases)},\n")
    output.write("  \"classes\": [\n")
    for index, case in enumerate(best_cases):
        comma = "," if index + 1 < len(best_cases) else ""
        output.write(f"    {json.dumps(case, separators=(',', ':'))}{comma}\n")
    output.write("  ]\n")
    output.write("}\n")

assert len(best_cases) == 54
print(f"Saved {len(best_cases)} best orders to {best_cases_path}")

Saved 54 best orders to txt/6_simul_best_pin_orders.json


## 8. Four-total-variable learning order and inspection flowchart

This selects the classes whose two conditions contain four variables total. It orders them by new scramble coverage after overlap, using all eight orientations, and saves a rotation-aware inspection flowchart in its own folder.

In [ ]:
exec(
    Path("four_variable_learning_order.py").read_text(encoding="utf-8"),
    globals(),
)

## 9. Maximum-six-total-variable learning order and inspection flowchart

This selects the classes whose two saved conditions contain at most six variable occurrences total across both equations; repeated variables are counted repeatedly. For recognition, every four-variable condition is displayed with two variables on each side, using `A to B` for the clockwise distance from dial A to dial B. It orders the classes by new scramble coverage after overlap using all eight orientations, and saves the results and inspection flowchart in a new folder.

In [ ]:
exec(
    Path("six_variable_learning_order.py").read_text(encoding="utf-8"),
    globals(),
)

## 10. Maximum-eight-term learning order and inspection flowchart

This selects classes whose two saved conditions contain at most eight nonzero terms total. A coefficient such as `2c` counts as one term. It uses the same distance-based recognition notation, optimizes new scramble coverage across all eight orientations, and saves the results in a separate folder.

In [ ]:
exec(
    Path("eight_variable_learning_order.py").read_text(encoding="utf-8"),
    globals(),
)